# Week 3, Classification: Counting with Weights

**What you'll do today.** You'll train a transparent classifier, a logistic regression, on a
pre-labeled pop corpus, then **read its mind**: the signed weights that are its most positive
and most negative words.

> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder; Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.


In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# Fetch the pinned package lists if they aren't beside the notebook (this happens
# whenever you open a single notebook straight from GitHub in Colab).
import os, urllib.request
_RAW = "https://raw.githubusercontent.com/lucianli123/culture-as-data-2026/main/notebooks/"
for _f in ("requirements.txt", "constraints.txt"):
    if not os.path.exists(_f):
        urllib.request.urlretrieve(_RAW + _f, _f)

# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q pandas scikit-learn matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
print("imports ok")

In [ ]:
import os

## A real labeled corpus: two subreddits

The question classification answers best: **which community does a post come from, and
which words give it away?** In class we load two subreddits live (nothing is stored in
the repo; community text is analyze-only). If the live load is unavailable, the notebook
falls back to two novelists fetched live from Gutenberg. The mechanics are identical, and
both are real.

In [ ]:
import re

SUB_A, SUB_B = "askscience", "gaming"   # the two communities to tell apart

def load_subreddits(a, b, n=200):
    """Two subreddit slices, streamed live from a public mirror (analyze-only)."""
    from itertools import islice
    from datasets import load_dataset
    texts, labels = [], []
    for sub in (a, b):
        ds = load_dataset("parquet", split="train", streaming=True,
            data_files=f"hf://datasets/HuggingFaceGECLM/REDDIT_comments/data/{sub}-*.parquet")
        stream = (r["body"] for r in ds)
        good = (t for t in stream if isinstance(t, str) and 30 < len(t) < 300
                and "[removed]" not in t and "[deleted]" not in t)
        got = list(islice(good, n))
        texts += got; labels += [sub] * len(got)
    return pd.DataFrame({"text": texts, "label": labels})

def load_gutenberg(url):
    """Fetch from Project Gutenberg and strip the boilerplate."""
    import requests
    raw = requests.get(url, timeout=30).text.replace("\r\n", "\n")
    body = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw, flags=re.S)[-1]
    return re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", body)[0]

def load_novelists(n=400):
    """The fallback pair: Shelley against Stoker, fetched live from Gutenberg."""
    def sents(url):
        t = load_gutenberg(url)
        return [x.strip().replace("\n", " ") for x in re.split(r"(?<=[.!?])\s+", t) if 40 < len(x) < 180][:n]
    a = sents("https://www.gutenberg.org/cache/epub/84/pg84.txt")
    b = sents("https://www.gutenberg.org/cache/epub/345/pg345.txt")
    return pd.DataFrame({"text": a + b, "label": ["shelley"] * len(a) + ["stoker"] * len(b)})

try:
    df = load_subreddits(SUB_A, SUB_B)
    LABEL_A, LABEL_B = SUB_A, SUB_B
    print(f"live corpus: r/{SUB_A} vs r/{SUB_B}")
except Exception as e:
    print("live subreddit load unavailable, using the novelist fallback:", type(e).__name__)
    df = load_novelists()
    LABEL_A, LABEL_B = "shelley", "stoker"
print(df["label"].value_counts().to_string())
df.sample(4, random_state=1)

## Train it, the AI writes this; reading the result is your job

Each word becomes a column, the model learns a weight (a vote, for or against), and it adds them
up.

In [ ]:
vec = CountVectorizer()
X = vec.fit_transform(df["text"])
y = (df["label"] == LABEL_A).astype(int)

# Hold out a quarter of the rows the model never sees; accuracy is measured there:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("held-out accuracy:", round(accuracy_score(yte, clf.predict(Xte)), 3))
clf_full = LogisticRegression(max_iter=1000).fit(X, y)  # refit on all for weight-reading

## Read its mind, the signed weights

The most **positive** and most **negative** words are the model's mind on the table.

In [ ]:
terms = np.array(vec.get_feature_names_out())
w = clf_full.coef_.ravel()
order = w.argsort()
print("Most NEGATIVE words:", ", ".join(terms[order[:6]]))
print("Most POSITIVE words:", ", ".join(terms[order[-6:][::-1]]))

plt.figure(figsize=(6,3))
top = np.concatenate([order[:6], order[-6:]])
plt.barh(terms[top], w[top])
plt.title("The classifier's signed weights")
plt.tight_layout(); plt.show()

### Build it a "black cat"

The Teachable Machine demo learned a bias from a skewed training set (orange cats, brown dogs).

In [ ]:
def predict(s):
    p = clf_full.predict_proba(vec.transform([s]))[0, 1]
    return f"{LABEL_A if p > 0.5 else LABEL_B} (p_{LABEL_A}={p:.2f})"

for label in (LABEL_A, LABEL_B):
    example = df[df["label"] == label]["text"].iloc[-1]
    print(f"held-out-ish {label!r:14}:", predict(example))
print("the black cat (Shakespeare - neither class):",
      predict("shall i compare thee to a summer's day"))
# The model has two boxes and no concept of "neither." Every classifier you will ever
# meet has this property; the question is only how confidently it hides it.

### The words that give each author away

Before the caveats: the stylistic fingerprint the machine learned, on the table.

## Playground: your question

The classifier is a kit too. Choose your own two classes: any two corpora, or two
slices of one (early sonnets against late? Frankenstein against Dracula?), retrain, and
read the weights. Write the question first.

In [ ]:
# MY QUESTION: ...one sentence here...
# Any two piles of real text work: two other subreddits (edit SUB_A/SUB_B above and
# rerun), two novelists, early sonnets against late. Edit these four values:
_nov = load_novelists()
CLASS_A, NAME_A = list(_nov[_nov.label == "shelley"]["text"]), "shelley"
CLASS_B, NAME_B = list(_nov[_nov.label == "stoker"]["text"]), "stoker"

df2 = pd.DataFrame({"text": CLASS_A + CLASS_B, "label": [NAME_A]*len(CLASS_A) + [NAME_B]*len(CLASS_B)})
vec2 = CountVectorizer()
X2 = vec2.fit_transform(df2["text"]); y2 = (df2["label"] == NAME_A).astype(int)
Xtr2, Xte2, ytr2, yte2 = train_test_split(X2, y2, test_size=0.25, random_state=0, stratify=y2)
clf2 = LogisticRegression(max_iter=2000).fit(Xtr2, ytr2)
print(f"holdout accuracy: {clf2.score(Xte2, yte2):.2f}")
t2 = np.array(vec2.get_feature_names_out()); w2 = clf2.coef_.ravel(); o2 = w2.argsort()
print(f"most {NAME_B}-flavored:", ", ".join(t2[o2[:6]]))
print(f"most {NAME_A}-flavored:", ", ".join(t2[o2[-6:][::-1]]))